# TXF Overnight → 0050 Open v0.1.4

Purpose: practical trading reference from historical TXF overnight returns and FinLab adjusted 0050 prices.

Smoke test (synthetic): `MODE='smoke'`. Phase B QA: `validation` remains descriptive only and bounded to 100 days. Phase C real-data historical run: set `MODE='full'`, deliberately select the years to download, and set `CONFIRM_FULL=True`.

TXF Adaptive Close uses the 13:45/05:00 minute close if a valid bar exists, otherwise exact 13:44/04:59. No fill, same contract month required. Expiring-contract day 13:29 is excluded when followed by a different-month night.

Source history limits and exchange-shifted expiry days may need explicit review; scheduled third Wednesday is not exchange-confirmed for every year.

Output first read: reports/summary.md, tables/trading_reference.csv, trading_reference_by_year.csv, trading_reference_holdout.csv, close_policy_comparison.csv. No profitable-trade inference.

## Statistics-only Resume (no market data downloads)

Select `MODE='resume_statistics'` to replay the exact previously uploaded
FAILED STATISTICS run using `SOURCE_RUN_FOLDER_ID` and `SOURCE_EXPECTED_COMMIT`.
The notebook validates its Drive folder ID and original Git SHA, downloads
only the original event dataset, config, manifest and run_info, then executes
the SAME shared statistical functions into a new timestamped run folder.
No FinLab/Fubon credentials or wheel are needed. Old output and caches stay
untouched. Source price/contract/minute QA is NOT re-run: its prior event
classifications and exclusion reasons are used as-is.


In [ ]:
MODE = 'smoke'  # smoke / pilot / validation / full / resume_statistics
START = '2026-09-01'  # Phase C example: '2018-01-01'; full requires CONFIRM_FULL
END = '2026-09-07'    # Phase C example: '2026-09-30'
CONFIRM_FULL = False  # IMPORTANT: set True explicitly only after choosing date range
CLOSE_POLICY = 'adaptive' # main convention; 'strict' for old quarantine only
REQUEST_SPACING_SECONDS = 1.1
FUBON_WHEEL = ''  # optional verified local wheel override
FOLDER_ID = '1E9VkK_NgePZUidKu6ZqB2gceLnKy9kYx'
SOURCE_RUN_FOLDER_ID = '1ouocV-NPdL9-cH14ORbRpDTabCPVqSTc'  # Exact archived FAILED STATISTICS Drive folder; resume mode only
SOURCE_EXPECTED_COMMIT = '89ab18d248dbb5a66b75f950973a8fca43cbd74e'


In [ ]:
import os, sys, subprocess, pathlib, tempfile
from google.colab import drive, userdata, auth
drive.mount('/content/drive')
repo = pathlib.Path('/content/txf_overnight_spot_open')
# Token 經 askpass 傳遞，不放 URL、命令列或 git remote。
try:
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = ''
with tempfile.TemporaryDirectory() as td:
    ask = pathlib.Path(td) / 'askpass.py'
    ask.write_text('#!/usr/bin/env python3\nimport os,sys\nprint("x-access-token" if "Username" in sys.argv[1] else os.environ.get("TXF_CLONE_TOKEN", ""))\n')
    ask.chmod(0o700)
    env = dict(os.environ, GIT_ASKPASS=str(ask), GIT_TERMINAL_PROMPT='0', TXF_CLONE_TOKEN=token or '')
    if repo.exists():
        clean = subprocess.check_output(['git','-C',str(repo),'status','--porcelain'],text=True)
        remote = subprocess.check_output(['git','-C',str(repo),'remote','get-url','origin'],text=True).strip()
        branch = subprocess.check_output(['git','-C',str(repo),'branch','--show-current'],text=True).strip()
        if clean or remote != 'https://github.com/hh4832/txf_overnight_spot_open.git' or branch != 'main':
            raise RuntimeError('Existing checkout differs or is dirty; preserve it and resolve first')
        cmd = ['git','-C',str(repo),'pull','--ff-only','origin','main']
    else:
        cmd = ['git','clone','--branch','main','https://github.com/hh4832/txf_overnight_spot_open.git',str(repo)]
    result = subprocess.run(cmd,env=env,capture_output=True)
    if result.returncode:
        raise RuntimeError('GitHub clone/update failed; verify repository access and Secret')
del token, env
os.chdir(repo)
# Colab retains imports after git pull; never silently run stale code.
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
if any(name in sys.modules for name in ('src.pipeline', 'src.config', 'src.build_event_dataset')):
    raise RuntimeError('Repository may have updated after import: restart Colab runtime and Run All')
if sys.version_info < (3,11): raise RuntimeError('Python >=3.11 required')
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'],check=True)
print('Git SHA:', subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())


In [ ]:
# Authenticate Drive before attempting the private SDK download.
from google.colab import userdata, auth
auth.authenticate_user()
import google.auth
from googleapiclient.discovery import build
credentials, _ = google.auth.default()
service = build('drive', 'v3', credentials=credentials, cache_discovery=False)

if MODE not in ('smoke','resume_statistics'):
    from src.sdk_install import ensure_fubon_sdk
    sdk_version = ensure_fubon_sdk(service=service, override=FUBON_WHEEL)
    print('Fubon SDK ready:', sdk_version)
    for name in ('FINLAB_API_TOKEN','FUBON_PERSONAL_ID','FUBON_API_KEY','FUBON_CERT_PATH','FUBON_CERT_PASSWORD'):
        try:
            value = userdata.get(name)
        except Exception:
            raise RuntimeError(name + ': unavailable') from None
        if not value:
            raise RuntimeError(name + ': missing')
        os.environ[name] = value
    del value

In [ ]:
from src.drive_io import validate_folder, verified_cache_root, upload_run
validate_folder(service,FOLDER_ID)
cache_root = verified_cache_root(service,FOLDER_ID,'/content/drive/MyDrive')
print('Target Folder ID verified:', FOLDER_ID)

In [ ]:
from src.drive_io import download_statistics_resume_source
from src.config import load
from src.pipeline import run
subprocess.run([sys.executable,'-m','pytest','-q'],check=True)
# Research results are staged locally; raw vendor cache remains in verified Drive.
output = pathlib.Path('/content/txf_research_outputs')
before = set((output/'runs').glob('*'))
try:
    if MODE == 'resume_statistics':
        from src.resume_statistics import resume
        # No FinLab login or Fubon request. Download the four source files
        # from the exact previously FAILED run folder ID, not guessed paths.
        with tempfile.TemporaryDirectory() as temporary_source:
            source = download_statistics_resume_source(
                service,FOLDER_ID,SOURCE_RUN_FOLDER_ID,temporary_source)
            run_dir = resume(
                source,output,expected_source_sha=SOURCE_EXPECTED_COMMIT)
    else:
        if MODE == 'validation':
            assert 'validation' in pathlib.Path('src/config.py').read_text(), 'Stale validation mode'
        cfg = load(mode=MODE,start=START,end=END,confirm_full=CONFIRM_FULL,
                   request_spacing_seconds=REQUEST_SPACING_SECONDS,
                   close_policy=CLOSE_POLICY)
        run_dir = run(cfg,output,cache_root=cache_root)
except Exception:
    failed = set((output/'runs').glob('*')) - before
    for folder in failed:
        upload_run(service,FOLDER_ID,folder)
    raise
uploaded_id = upload_run(service,FOLDER_ID,run_dir)
print('Completed:', MODE, 'https://drive.google.com/drive/folders/' + uploaded_id)
from IPython.display import Markdown, display
display(Markdown((run_dir/'reports/summary.md').read_text()))
